# Comparing this pipeline against just prompting an LLM

The rest of this session built a synthetic customer table the deliberate way: a real table, checked and typed by `Preprocessor`, then a generative model (CTGAN or TVAE) trained and auto-tuned specifically on it. This notebook asks the obvious shortcut question: what if you just described the table to a large language model and asked it to make one up?

This is a real option a marketing team actually has, not a hypothetical, and it's worth testing on the same task, against the same real, held-out customers, rather than assumed to be worse (or good enough) without checking.

**Time**: about 15-20 minutes, depending on how many rows you generate
**Cost**: a small number of Gemini API calls, at Gemini's Flash-Lite pay-as-you-go rate (fractions of a cent per request), see the model-selection note further down before raising the row count much past this notebook's default

> **Disclaimer.** This is one prompting strategy, tested once, against one target column, on one dataset. It's a real comparison, not an exhaustive one, and a different prompt, a different model, or a different target could land differently. Treat the result as a data point about this approach on this task, not a general verdict on LLM-generated data.

## Before this notebook: run the main pipeline notebook first

This notebook loads `preprocessed_real_table.csv`, which `synthetic_data_pipeline.ipynb` saves once it finishes preprocessing. Run that notebook up through its preprocessing section at least once, in this same folder, before running this one; otherwise the load cell below will fail with a clear message telling you what's missing.

Reusing that exact file, rather than re-deriving the table from the raw Olist data again here, is what keeps this comparison fair: both notebooks end up testing against the exact same real, held-out customers.

## Setting up your environment

Same split as every other notebook this session: nothing to do beforehand in Colab, or one `uv` setup in a terminal first if running locally. Full walkthrough of `uv` is in [Session 1's setup guide](../session_1/setup_guide.ipynb).

**Locally**, before opening this notebook:
```bash
cd session_4
uv venv
uv pip install -r requirements.txt
```

This notebook also needs a Gemini API key, the same one used in Session 1. **In Colab**: Secrets (🔑 icon) → add `GEMINI_API_KEY`, with the notebook access toggle on. **Locally**: a `.env` file in `session_4/` containing `GEMINI_API_KEY=your_key_here` (and confirm `.env` is in `.gitignore`, as Session 3's key-security lesson covers). If you don't already have a key, [Session 1's setup guide](../session_1/setup_guide.ipynb) walks through getting one, including the billing requirement.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab: explicit package list, not session_4/requirements.txt. "Open in
    # Colab" only loads this one file, so there's no repo alongside it to read
    # that file from. Keep this list in sync with requirements.txt by hand.
    %pip install -q google-genai pydantic pandas numpy scikit-learn xgboost python-dotenv
else:
    # Local: installs from session_4/requirements.txt, looked up by directory
    # for the same reason the main pipeline notebook's setup cell explains.
    import pathlib
    _cwd = pathlib.Path.cwd()
    if _cwd.name == "session_4" and (_cwd / "requirements.txt").exists():
        _req_path = "requirements.txt"
    elif (_cwd / "session_4" / "requirements.txt").exists():
        _req_path = "session_4/requirements.txt"
    elif (_cwd.parent / "session_4" / "requirements.txt").exists():
        _req_path = "../session_4/requirements.txt"
    else:
        _req_path = None

    if _req_path is None:
        print("Couldn't find session_4/requirements.txt from the current working directory:", _cwd)
        print("Set up your local environment first, see the markdown cell above for the uv commands.")
    else:
        %pip install -q -r {_req_path}

print("Package install step finished.")

In [ ]:
# Loads the key from Colab Secrets or a local .env file, and only ever prints
# a truncated preview, never the full key, the same rule Session 3's
# key-security lesson asks for at every point a key gets handled.
from google import genai

if IN_COLAB:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
else:
    from dotenv import load_dotenv
    import os
    load_dotenv()
    api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("No key found. In Colab: add GEMINI_API_KEY to Secrets and turn its access toggle on.")
    print("Locally: add GEMINI_API_KEY=your_key_here to a .env file in session_4/.")
else:
    print("Key loaded. Starts with:", api_key[:10] + "...")
    client = genai.Client(api_key=api_key)

## Loading the same real table this session already built

`preprocessed_real_table.csv` is the exact table CTGAN and TVAE trained on earlier in this session, after `Preprocessor` cleaned it. Splitting it the same way, with the same `random_state`, means `real_test` here is the identical set of held-out customers the main notebook already tested CTGAN and TVAE against, so the comparison this notebook builds toward is against those same numbers, not a fresh, slightly different test set.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import xgboost as xgb
import pathlib

CSV_PATH = pathlib.Path("preprocessed_real_table.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(
        "preprocessed_real_table.csv not found. Run synthetic_data_pipeline.ipynb "
        "first, at least through its preprocessing section, it saves this file "
        "as one of its last steps there. See the notebook's second cell for why."
    )

preprocessed = pd.read_csv(CSV_PATH)

CAT_COLS = ["region", "primary_payment_type", "top_category"]
NUM_COLS = ["num_orders", "avg_basket_value", "days_since_last_order"]
TARGET = "bought_again_90d"

# Same random_state as the main notebook's suitability test, so real_test here
# is the identical held-out set CTGAN and TVAE were already tested against.
real_train, real_test = train_test_split(
    preprocessed, test_size=0.25, stratify=preprocessed[TARGET], random_state=42
)

print("real_train:", real_train.shape, " positive rate:", real_train[TARGET].mean())
print("real_test: ", real_test.shape, " positive rate:", real_test[TARGET].mean())

## What gets sent to the model, and what doesn't

CTGAN and TVAE never leave this machine (or Colab's own runtime) during training: the real customer table stays local the entire time. Prompting an LLM is different: whatever goes into the prompt travels to a third party's API. That difference matters enough to be deliberate about it here, not just convenient.

This notebook sends the LLM a description of the table, not the table itself: column names, what each one means, and summary statistics, category proportions, numeric ranges and averages, computed from `real_train`. No individual customer's row is ever included in a prompt. It's the same instinct as reading a log before sending it, earlier this session: check what's actually leaving before it leaves, not after.

In [ ]:
# Category proportions and numeric summaries computed from real_train only,
# this dict is literally what gets embedded in the prompt below, so printing it
# here is a direct, checkable answer to "what did we actually send the model?"
summary = {"target_positive_rate": round(real_train[TARGET].mean(), 4)}

for col in CAT_COLS:
    proportions = real_train[col].value_counts(normalize=True).round(4)
    summary[col] = proportions.to_dict()

for col in NUM_COLS:
    summary[col] = {
        "mean": round(real_train[col].mean(), 2),
        "std": round(real_train[col].std(), 2),
        "min": round(real_train[col].min(), 2),
        "max": round(real_train[col].max(), 2),
    }

import json as _json
print(_json.dumps(summary, indent=2))

## Asking the model for synthetic rows

Rather than parse free-text back into a table, this uses the Gemini API's structured output: a schema (built with a small `pydantic` model) that tells Gemini exactly what shape each row has to come back in, so the result is already a usable table, not something to regex out of a paragraph.

One call returns one batch of rows, not a full table, since there's a practical limit to how much any single request should ask a model to generate reliably in one pass. This notebook loops over several batches instead, building the table up the same way you'd expect from an API with per-request limits, and stopping once it reaches roughly the same row count as `real_train`, so the two approaches are compared at a similar scale.

**On model choice**: this defaults to `gemini-2.5-flash-lite`, the same lightweight, low-cost model this course defaults to elsewhere. Structured generation like this is a reasonable task for it, but if the returned rows look repetitive or off (check the printed sample after the first batch), `gemini-2.5-flash` is the next step up in quality, at a higher per-request cost. Confirm current pricing for either at [aistudio.google.com](https://aistudio.google.com) before raising `TOTAL_ROWS` much.

In [ ]:
from pydantic import BaseModel

class SyntheticCustomer(BaseModel):
    region: str
    primary_payment_type: str
    top_category: str
    num_orders: int
    avg_basket_value: float
    days_since_last_order: int
    bought_again_90d: int  # 0 or 1

MODEL_NAME = "gemini-2.5-flash-lite"
BATCH_SIZE = 25
# Capped well under len(real_train) on purpose: this is a teaching notebook,
# and every extra row here is another slice of an API call's cost and time.
# Raise this once you've looked at the first batch and trust the output.
TOTAL_ROWS = min(len(real_train), 300)
N_BATCHES = -(-TOTAL_ROWS // BATCH_SIZE)  # ceiling division

PROMPT_TEMPLATE = f"""You are generating synthetic e-commerce customer data for a
data science teaching exercise. Nothing about these rows describes a real person,
generate plausible, varied rows that are statistically consistent with the summary
below, not a copy of any real customer.

Columns:
- region: a Brazilian state abbreviation
- primary_payment_type: the payment method used most often by this customer
- top_category: the product category this customer buys most often
- num_orders: number of orders placed before the prediction cutoff (positive integer)
- avg_basket_value: average order value in Brazilian reais (positive number)
- days_since_last_order: days between the customer's last order and the cutoff (non-negative integer)
- bought_again_90d: 1 if the customer bought again within 90 days after the cutoff, else 0

Summary statistics from the real data this should resemble (category proportions,
and mean/std/min/max for numeric columns):
{_json.dumps(summary, indent=2)}

Generate exactly {{batch_size}} rows as a JSON list matching the schema. Vary the
values realistically across rows rather than repeating the averages. Roughly
{{target_rate:.1%}} of rows should have bought_again_90d = 1, matching the summary above.
"""

synthetic_rows = []
for batch_num in range(N_BATCHES):
    this_batch_size = min(BATCH_SIZE, TOTAL_ROWS - len(synthetic_rows))
    if this_batch_size <= 0:
        break
    prompt = PROMPT_TEMPLATE.format(batch_size=this_batch_size, target_rate=summary["target_positive_rate"])
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config={"response_mime_type": "application/json", "response_schema": list[SyntheticCustomer]},
        )
        batch_rows = response.parsed
        synthetic_rows.extend(batch_rows)
        print(f"batch {batch_num + 1}/{N_BATCHES}: got {len(batch_rows)} rows (running total: {len(synthetic_rows)})")
    except Exception as e:
        # One failed batch (a network blip, a parsing miss) shouldn't sink the
        # whole run: skip it, note it, and keep going with what's left.
        print(f"batch {batch_num + 1}/{N_BATCHES}: failed ({type(e).__name__}: {e}), skipping")

llm_synthetic_table = pd.DataFrame([row.model_dump() for row in synthetic_rows])
llm_synthetic_table.to_csv("llm_synthetic_table.csv", index=False)
print("\nfinal LLM-generated table:", llm_synthetic_table.shape)
llm_synthetic_table.head()

## Why this table is smaller, and why that's part of the answer

CTGAN and TVAE spend their time up front, during training, and then generate rows almost instantly afterward: the notebook earlier this session asked for `len(preprocessed)` rows in one line and got them back at once. An LLM prompted this way spends its time per row requested, in per-call batches with their own cost and latency each. Scaling this approach up to the full table's size means scaling up the number of calls, and therefore the cost and the wait, roughly linearly. That's a real, structural difference worth weighing alongside whatever the accuracy comparison below shows: a slightly weaker but effectively free, instantly-resampleable table is a different trade than a slightly stronger one that costs more, row for row, every time you need more of it.

## The actual test: does a model trained on this data work?

Same test as the rest of this session: train the propensity model once on `real_train`, once on the LLM-generated table, and check both against the same `real_test` customers neither model has seen.

In [ ]:
def make_pipeline():
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ("num", "passthrough", NUM_COLS),
    ])
    clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, eval_metric="logloss", n_jobs=1)
    return Pipeline([("pre", pre), ("clf", clf)])

def evaluate(train_df, label):
    pipe = make_pipeline()
    pipe.fit(train_df[CAT_COLS + NUM_COLS], train_df[TARGET])
    preds = pipe.predict(real_test[CAT_COLS + NUM_COLS])
    metrics = {
        "accuracy": accuracy_score(real_test[TARGET], preds),
        "precision": precision_score(real_test[TARGET], preds, zero_division=0),
        "recall": recall_score(real_test[TARGET], preds, zero_division=0),
        "f1": f1_score(real_test[TARGET], preds, zero_division=0),
    }
    print(
        f"{label:22s} accuracy={metrics['accuracy']:.3f}  "
        f"precision={metrics['precision']:.3f}  "
        f"recall={metrics['recall']:.3f}  "
        f"f1={metrics['f1']:.3f}"
    )
    return metrics

real_metrics = evaluate(real_train, "trained on REAL")
llm_metrics = evaluate(llm_synthetic_table, "trained on LLM-GENERATED")

In [ ]:
# Same plain-language pattern the main pipeline notebook uses: react to whatever
# REAL and LLM-GENERATED actually scored, rather than assuming one outcome.
if real_metrics["f1"] == 0:
    print("Even the REAL-trained model found essentially nothing here. That's a property")
    print("of this task at this sample size, not something either synthetic approach")
    print("could have fixed, this run's comparison can't say much either way.")
elif llm_metrics["recall"] == 0:
    print(f"REAL found some signal (recall={real_metrics['recall']:.3f}, f1={real_metrics['f1']:.3f}).")
    print("LLM-GENERATED found none (recall = 0.000): a model trained on it predicts")
    print("\"no\" for every customer. On this run, this LLM-generated table did not")
    print("transfer for the actual task it was built for.")
elif llm_metrics["f1"] >= real_metrics["f1"] * 0.8:
    print(f"LLM-GENERATED (f1={llm_metrics['f1']:.3f}) came close to, or matched, REAL")
    print(f"(f1={real_metrics['f1']:.3f}) on this run, worth taking seriously as an option")
    print("for this task, at this row count and prompt.")
else:
    gap = real_metrics["f1"] - llm_metrics["f1"]
    print(f"LLM-GENERATED (f1={llm_metrics['f1']:.3f}) underperformed REAL (f1={real_metrics['f1']:.3f})")
    print(f"by {gap:.3f} f1: partial transfer, some signal, less of it than training on")
    print("real data directly.")

## What this comparison does, and doesn't, tell you

A close result here says a Gemini Flash-Lite model, prompted this way, with these summary statistics, on this target column, produced rows that transfer reasonably well to real customers. It says nothing about a different target, a different prompt, a larger row count, or a different model, the same boundary the main pipeline notebook named for CTGAN and TVAE applies here too, just for a different method.

What it does say something about, regardless of which way the accuracy numbers land: CTGAN and TVAE are trained specifically on this table's actual joint distribution, learning how columns relate to each other from the data itself, then sampling from what they learned. An LLM prompted this way is working from a written description of that distribution, in a single-column-summary form that can't fully capture how columns move together, filled in by whatever the model already knows about e-commerce data in general. That difference, not just whichever score happened to be higher on this run, is the more durable reason to expect one approach or the other to fit a given real task.

**Also in Session 4**: when to use synthetic data at all, how a synthetic dataset actually gets made with a dedicated platform, and what its metrics do and don't protect, this notebook assumes all three.

Questions belong in the Circle community.